# Attention Home Extractor

This notebook loads the latest `attention_home_1d` payload directly from the materialized pipeline snapshot and formats the homepage text without going through the Streamlit dashboard.

If the query fails because Azure credentials are missing, run `az login --use-device-code` in a terminal and rerun the cells.

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path


def find_app_root() -> Path:
    candidates = [
        Path.cwd(),
        Path("/home/azureuser/cloudfiles/code/Users/omai.r/spectral_nature/streamlit_alpaca_app"),
    ]
    for candidate in candidates:
        resolved = candidate.resolve()
        for path in [resolved, *resolved.parents]:
            if (path / "infra" / "deployment.outputs.env").exists() and (path / "data_access").exists():
                return path
    raise RuntimeError("Could not locate streamlit_alpaca_app root.")


APP_ROOT = find_app_root()
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

APP_ROOT

PosixPath('/mnt/batch/tasks/shared/LS_root/mounts/clusters/spectral-nature3/code/Users/omai.r/spectral_nature/streamlit_alpaca_app')

In [2]:
DEPLOYMENT_ENV_PATH = APP_ROOT / "infra" / "deployment.outputs.env"


def load_env_file(path: Path) -> dict[str, str]:
    values: dict[str, str] = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        clean = line.strip()
        if not clean or clean.startswith("#") or "=" not in clean:
            continue
        key, value = clean.split("=", 1)
        values[key.strip()] = value.strip()
    return values


deployment_env = load_env_file(DEPLOYMENT_ENV_PATH)

# These are sufficient for reading the materialized pipeline snapshots.
os.environ.setdefault("AZURE_STORAGE_ACCOUNT_URL", deployment_env.get("STORAGE_URL", ""))
os.environ.setdefault("AZURE_STORAGE_CONTAINER", "datasets")
os.environ.setdefault("AZURE_CLIENT_ID", deployment_env.get("MANAGED_IDENTITY_CLIENT_ID", ""))
os.environ.setdefault("AZURE_KEY_VAULT_NAME", deployment_env.get("KEYVAULT_NAME", ""))
os.environ.setdefault("KEY_VAULT_NAME", deployment_env.get("KEYVAULT_NAME", ""))
os.environ.setdefault("PIPELINE_RESOURCE_GROUP", deployment_env.get("RESOURCE_GROUP", ""))

configured_env = {
    key: os.environ.get(key, "")
    for key in [
        "AZURE_STORAGE_ACCOUNT_URL",
        "AZURE_STORAGE_CONTAINER",
        "AZURE_CLIENT_ID",
        "AZURE_KEY_VAULT_NAME",
        "KEY_VAULT_NAME",
        "PIPELINE_RESOURCE_GROUP",
    ]
}
configured_env

{'AZURE_STORAGE_ACCOUNT_URL': 'https://snpipeline03130136.blob.core.windows.net',
 'AZURE_STORAGE_CONTAINER': 'datasets',
 'AZURE_CLIENT_ID': '4f7c0e9a-2828-45cf-9153-86bbf52acf4e',
 'AZURE_KEY_VAULT_NAME': 'snpipelinekv03130136',
 'KEY_VAULT_NAME': 'snpipelinekv03130136',
 'PIPELINE_RESOURCE_GROUP': 'sn-pipeline-rg-03130136'}

In [3]:
from data_access.contracts import QueryRequest
from data_access.query_service import QueryService


FORCE_REFRESH = False

service = QueryService.from_environment()
response = service.execute(
    QueryRequest(
        operation="dataset",
        name="attention_home_1d",
        params={"force_refresh": FORCE_REFRESH},
    )
)

payload = response.payload
provenance = response.provenance.to_dict() if response.provenance is not None else {}

print("generated_at_utc:", payload.get("generated_at_utc"))
print("run_id:", payload.get("run_id"))
print("mode:", provenance.get("mode"))
print("datasets:", provenance.get("datasets"))

generated_at_utc: 2026-03-30T16:05:21.054458+00:00
run_id: bfbaef99-6c12-4289-a844-1f17d724a110
mode: materialized
datasets: ['attention_home_snapshots_1d']


In [4]:
from IPython.display import Markdown, display


def render_homepage_text(home_payload: dict[str, object]) -> None:
    generated_at = str(home_payload.get("generated_at_utc") or "")
    display(Markdown(f"## Homepage Snapshot\n\n- Generated: `{generated_at}`\n- Run ID: `{home_payload.get('run_id', '')}`"))

    top_events = list(home_payload.get("top_events") or [])
    must_read = list(home_payload.get("must_read_movers") or [])
    unresolved = list(home_payload.get("unresolved_large_moves") or [])

    display(Markdown(f"## Top Events Today\n\nCount: `{len(top_events)}`"))
    for index, event in enumerate(top_events, start=1):
        title = str(event.get("event_title") or f"Event {index}")
        what = str(event.get("what_happened_text") or "").strip()
        why = str(event.get("why_happened_text") or "").strip()
        affected = str(event.get("affected_assets_summary_text") or "").strip()
        display(
            Markdown(
                f"### {index}. {title}\n\n"
                f"**What Happened**\n\n{what}\n\n"
                f"**Why It Happened**\n\n{why}\n\n"
                f"**Affected Assets**\n\n{affected}"
            )
        )

    display(Markdown(f"## Must-Read Movers\n\nCount: `{len(must_read)}`"))
    for index, item in enumerate(must_read, start=1):
        symbol = str(item.get("symbol") or "")
        headline = str(item.get("headline") or symbol or f"Mover {index}")
        changed = str(item.get("what_changed_text") or "").strip()
        why = str(item.get("why_now_text") or "").strip()
        other = str(item.get("what_else_moved_text") or "").strip()
        display(
            Markdown(
                f"### {index}. {symbol} - {headline}\n\n"
                f"**What Changed Vs Expectation**\n\n{changed}\n\n"
                f"**Why Today**\n\n{why}\n\n"
                f"**What Else Moved**\n\n{other}"
            )
        )

    display(Markdown(f"## Unresolved Large Moves\n\nCount: `{len(unresolved)}`"))
    for index, item in enumerate(unresolved, start=1):
        symbol = str(item.get("symbol") or "")
        headline = str(item.get("headline") or symbol or f"Unresolved {index}")
        changed = str(item.get("what_changed_text") or "").strip()
        why = str(item.get("why_now_text") or "").strip()
        other = str(item.get("what_else_moved_text") or "").strip()
        display(
            Markdown(
                f"### {index}. {symbol} - {headline}\n\n"
                f"**What Changed Vs Expectation**\n\n{changed}\n\n"
                f"**Why Today**\n\n{why}\n\n"
                f"**What Else Moved**\n\n{other}"
            )
        )


render_homepage_text(payload)

## Homepage Snapshot

- Generated: `2026-03-30T16:05:21.054458+00:00`
- Run ID: `bfbaef99-6c12-4289-a844-1f17d724a110`

## Top Events Today

Count: `5`

### 1. Biotechnology weaker while pharma strength lifts broader Health Care

**What Happened**

Biotechnology names broadly traded lower with several development‑stage companies under pressure, while strength in large pharmaceutical companies helped offset the weakness and keep the broader Health Care sector supported.

**Why It Happened**

Kodiak Sciences shares rose after the company reported positive Phase 3 GLOW2 results for Zenkuda in diabetic retinopathy.

**Affected Assets**

The divergence pushed capital toward large-cap pharmaceutical exposure while leaving smaller clinical-stage developers more sensitive to idiosyncratic news and sentiment shifts around trial outcomes and pipeline expectations.

### 2. Unity guidance boost contrasts with broader pressure on tech and online platforms

**What Happened**

Online platform names within communication services traded lower while Unity surged after updating its first‑quarter revenue guidance, creating a split between company‑specific upside and broader tech weakness.

**Why It Happened**

Company-specific guidance drove Unity higher as improved revenue expectations lifted outlooks for its game development platform business. At the same time, rising long‑term Treasury yields tightened valuation pressure on growth stocks and coincided with the Nasdaq 100 entering correction territory, weighing on internet and cloud software names.

**Affected Assets**

The dynamic spilled into other growth‑oriented technology groups, where cloud software and internet platform companies faced broader risk‑off pressure even as select names with positive corporate updates diverged.

### 3. Gold Miners Gain as Precious-Metal Exposure Draws Flows

**What Happened**

Shares of gold mining and royalty companies broadly advanced, lifting the precious‑metals segment within materials as investors rotated into firms whose revenues are tied to gold prices.

**Why It Happened**

Demand for gold-linked equities increased as investors sought exposure to the metal’s price leverage, which tends to amplify moves in bullion through operating margins and royalty revenue streams. Because miners’ costs are relatively fixed in the short term, incremental strength or expected strength in gold prices can translate into disproportionate earnings sensitivity, drawing equity flows into the group.

**Affected Assets**

The move also supported related precious‑metal exposure such as royalty and streaming companies, while the broader materials sector saw some sympathetic strength as commodity‑linked equities attracted incremental capital.

...

### 1. EEIQ - EEIQ declines as post–reverse split trading volatility continues

**What Changed Vs Expectation**

EEIQ fell sharply today relative to its recent baseline.

**Why Today**

The decline appears to extend trading volatility that often follows reverse stock splits. After EpicQuest Education consolidated its shares to regain Nasdaq’s minimum bid-price compliance, the higher post-split share price and reduced share count can change liquidity and trading dynamics, which can lead to sharper price swings as the market rebalances positions.

**What Else Moved**

Broader weakness across several consumer-facing and travel-related names, including cruise operators and restaurant chains, pointed to softer risk appetite in equities, which can spill over into smaller or less liquid companies and amplify downside moves.

### 2. ETR - ETR moves sharply today

**What Changed Vs Expectation**

ETR rose sharply today relative to its recent baseline.

**Why Today**

A clear company‑specific catalyst was not identified. The move occurred during a broad equity selloff as long‑term Treasury yields pushed toward 5%, a backdrop that can shift investor positioning and risk appetite and sometimes leads to relative demand for certain defensive or regulated businesses.

**What Else Moved**

The broader market weakened, with major technology-heavy indexes sliding into correction territory as higher long‑term borrowing costs pressured equity valuations and risk appetite.

### 3. NIB - NIB moves sharply today

**What Changed Vs Expectation**

NIB rose sharply today relative to its recent baseline.

**Why Today**

No clear same-day catalyst was identified for NIB.

**What Else Moved**

No clear same-day peer or cross-asset spillover was confirmed.

### 4. SATL - SATL pulls back as investors continue digesting filings

**What Changed Vs Expectation**

SATL fell sharply today relative to its recent baseline.

**Why Today**

The drop appears to extend a cooling phase after a strong run over the past week. As investors continue reviewing Satellogic’s recently filed annual results and governance updates, some of the earlier momentum is fading, prompting reassessment of valuation and near-term expectations.

**What Else Moved**

Moves elsewhere were mixed, with activity in adjacent engineering and infrastructure names such as AGX indicating that today’s action was not part of a broad sector swing but more company-specific digestion.

### 5. CRWD - CRWD falls sharply as analyst outlook shifts weigh on sentiment

**What Changed Vs Expectation**

CRWD fell sharply today relative to its recent baseline.

**Why Today**

Reports highlighted that several top Wall Street analysts changed their outlook on major technology names. When analysts revise views on widely followed growth stocks, it can prompt investors to reassess valuation and earnings expectations across the group, leading to selling pressure in high-multiple software and cybersecurity names like CrowdStrike.

**What Else Moved**

Other cloud and cybersecurity software companies also traded lower, including Datadog and Palo Alto Networks, suggesting the reaction spread across the software and security segment rather than being isolated to CrowdStrike.

In [5]:
from data_access.layer import DataAccessLayer


SELECTED_BUNDLE_ID = str((payload.get("top_events") or [{}])[0].get("bundle_id") or "")

data_access = DataAccessLayer()
bundle_resolved = data_access.resolve_attention_research_bundle(SELECTED_BUNDLE_ID, force_refresh=False)
bundle_payload = bundle_resolved.payload

print("bundle_id:", SELECTED_BUNDLE_ID)
print("bundle_mode:", bundle_resolved.provenance.mode if bundle_resolved.provenance else "")
print("title:", bundle_payload.get("title") or bundle_payload.get("event_title") or "")
print("what_happened_text:", bundle_payload.get("what_happened_text") or "")
print("why_happened_text:", bundle_payload.get("why_happened_text") or "")
print("affected_assets_summary_text:", bundle_payload.get("affected_assets_summary_text") or "")

bundle_id: event::cluster-02-95f0963cb9
bundle_mode: materialized
title: Biotechnology weaker while pharma strength lifts broader Health Care
what_happened_text: Biotechnology names broadly traded lower with several development‑stage companies under pressure, while strength in large pharmaceutical companies helped offset the weakness and keep the broader Health Care sector supported.
why_happened_text: Kodiak Sciences shares rose after the company reported positive Phase 3 GLOW2 results for Zenkuda in diabetic retinopathy.
affected_assets_summary_text: The divergence pushed capital toward large-cap pharmaceutical exposure while leaving smaller clinical-stage developers more sensitive to idiosyncratic news and sentiment shifts around trial outcomes and pipeline expectations.
